# Model

In [10]:
# [ESSENTIAL] Load cluster-aware train/val/test split from parquet
import gemmi, torch
import pandas as pd
import numpy as np

# Load just the two ID columns — no need to load coordinates


df_ids = pd.read_parquet('data/clusters/clusters.parquet', columns=['structure_id', 'cluster_id']) 

cluster_to_structures = df_ids.drop_duplicates().groupby('cluster_id')['structure_id'].apply(list).to_dict()
clusters = list(cluster_to_structures.keys())

# Split clusters (not structures) into train/val/test
rng = np.random.default_rng(42)
rng.shuffle(clusters)
n = len(clusters)
train_clusters = clusters[:int(0.8*n)]
val_clusters   = clusters[int(0.8*n):int(0.9*n)]
test_clusters  = clusters[int(0.9*n):]

# Flatten to structure IDs
train_ids = [s for c in train_clusters for s in cluster_to_structures[c]]
val_ids   = [s for c in val_clusters   for s in cluster_to_structures[c]]
test_ids  = [s for c in test_clusters  for s in cluster_to_structures[c]]

print(len(train_ids), len(val_ids), len(test_ids))

34968 3503 4880


In [11]:
# [ESSENTIAL] MetalBindingDataset — core dataset class with atom14 coords, k-NN graph, edge attributes, and cache support
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
import glob
import os
from scipy.spatial import KDTree
import torch
from torch_geometric.data import Data

RESIDUES   = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY',
              'HIS','ILE','LEU','LYS','MET','PHE','PRO','SER',
              'THR','TRP','TYR','VAL','UNK']
RES_TO_IDX = {r: i for i, r in enumerate(RESIDUES)}
DISTAL_CACHE_DIR = 'data/pt_cache_distal'

_ATOM14_COLS = [f'{c}{i}' for i in range(14) for c in ('x', 'y', 'z')]

#takes atom_14 coordinates and returns 
def _compute_distal(coords14_raw):
    """(N,14,3) with NaN for missing → (N,3) last-finite atom per residue."""
    #mask for nans
    finite_mask = np.isfinite(coords14_raw).all(axis=2)   # (N, 14)
    #gets the last index without a Nan
    last_idx    = (finite_mask * np.arange(14)).argmax(axis=1)
    #takes the alpha-carbon position coords for glycine
    last_idx[~finite_mask.any(axis=1)] = 1                # CA fallback
    return coords14_raw[np.arange(len(coords14_raw)), last_idx, :]

#class structure
class MetalBindingDataset(Dataset):
    def __init__(self, chunk_glob, structure_ids, k=10, cache_dir='data/pt_cache_v3',
                 esm_cache_dir=None):

        self.k = k
        self.ids = list(structure_ids)
        self.cache_dir = cache_dir
        self.esm_cache_dir = esm_cache_dir  # None → no ESM features; set to 'data/esm2_cache' for PLM mode

        self.index = {}
        for fpath in glob.glob(chunk_glob):
            df = pd.read_parquet(fpath, columns=['structure_id'])
            for sid in df['structure_id'].unique():
                self.index[sid] = fpath

    def __len__(self):
        return len(self.ids)

    def _process(self, idx):
        sid = self.ids[idx]
        fpath = self.index[sid]
        df = pd.read_parquet(fpath)
        df = df[df['structure_id'] == sid]

        protein = df[~df['is_metal']].drop(columns='is_metal').reset_index(drop=True)
        metals  = df[df['is_metal']].drop(columns='is_metal').reset_index(drop=True)

        # ── Atom14 coords (N, 14, 3) ──────────────────────────────────────────
        coords14_raw = protein[_ATOM14_COLS].values.reshape(-1, 14, 3)  # NaN intact

        # Filter: CA (atom 1) must be finite
        valid        = np.isfinite(coords14_raw[:, 1, :]).all(axis=1)
        coords14_raw = coords14_raw[valid]

        # Distal atom computed BEFORE zeroing NaN (needs NaN to find last real atom)
        pos_distal = torch.tensor(_compute_distal(coords14_raw), dtype=torch.float)

        # Zero-fill missing atoms for node features
        coords14 = np.nan_to_num(coords14_raw, nan=0.0)

        res_names = protein['res_name'].values[valid]
        res_type  = torch.tensor([RES_TO_IDX.get(r, RES_TO_IDX['UNK']) for r in res_names],
                                  dtype=torch.long)

        pos = torch.tensor(coords14[:, 1, :], dtype=torch.float)   # CA positions
        x   = torch.tensor(coords14.reshape(len(coords14), -1), dtype=torch.float)  # (N, 42)

        # ── Graph: k-NN on CA ─────────────────────────────────────────────────
        ca = coords14[:, 1, :]
        tree = KDTree(ca)
        _, neighbors = tree.query(ca, k=self.k + 1)
        neighbors = neighbors[:, 1:]

        src = np.repeat(np.arange(len(coords14)), self.k)
        dst = neighbors.flatten()

        # ── 4-dim edge attributes ─────────────────────────────────────────────
        ca_src  = ca[src];               ca_dst  = ca[dst]
        dis_np  = pos_distal.numpy()
        dis_src = dis_np[src];           dis_dst = dis_np[dst]

        d_ca_ca   = np.linalg.norm(ca_src  - ca_dst,  axis=1)
        d_dis_dis = np.linalg.norm(dis_src - dis_dst, axis=1)
        d_cross   = (np.linalg.norm(ca_src  - dis_dst, axis=1)
                   + np.linalg.norm(dis_src - ca_dst,  axis=1)) / 2

        v_src = dis_src - ca_src;  v_dst = dis_dst - ca_dst
        n_src = np.linalg.norm(v_src, axis=1, keepdims=True).clip(1e-8)
        n_dst = np.linalg.norm(v_dst, axis=1, keepdims=True).clip(1e-8)
        sc_dot = ((v_src / n_src) * (v_dst / n_dst)).sum(axis=1)

        edge_attr  = np.stack([d_ca_ca, d_dis_dis, d_cross, sc_dot], axis=1)  # (E, 4)
        edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long)

        y = torch.tensor(metals[['x0', 'y0', 'z0']].values, dtype=torch.float)

        return Data(
            x=x,
            edge_index=edge_index,
            edge_attr=torch.tensor(edge_attr, dtype=torch.float),
            y=y,
            num_metals=torch.tensor([y.shape[0]], dtype=torch.long),
            res_type=res_type,
            pos=pos,
            pos_distal=pos_distal,
        )

    def _attach_distal(self, d, sid):
        """Fallback: attach pos_distal from sidecar if cached file predates v3."""
        if hasattr(d, 'pos_distal'):
            return d
        distal_path = os.path.join(DISTAL_CACHE_DIR, f'{sid}.pt')
        if os.path.exists(distal_path):
            try:
                pd_ = torch.load(distal_path, weights_only=True)
                if (pd_ is not None and isinstance(pd_, torch.Tensor)
                        and pd_.shape[0] == d.pos.shape[0]):
                    d.pos_distal = pd_
                    return d
            except Exception:
                pass
        d.pos_distal = d.pos  # last-resort: CA
        return d

    def _attach_esm(self, d, sid):
        """Attach ESM-2 per-residue embeddings if esm_cache_dir is set and cache exists."""
        if self.esm_cache_dir is None:
            return d
        esm_path = os.path.join(self.esm_cache_dir, f'{sid}.pt')
        if os.path.exists(esm_path):
            emb = torch.load(esm_path, weights_only=True)
            if emb.shape[0] == d.x.shape[0]:   # length guard
                d.esm = emb
        return d

    def __getitem__(self, idx):
        sid = self.ids[idx]
        cache_path = os.path.join(self.cache_dir, f'{sid}.pt')
        if os.path.exists(cache_path):
            d = torch.load(cache_path, weights_only=False)
            d.num_metals = torch.tensor([d.y.shape[0]], dtype=torch.long)
            d = self._attach_distal(d, sid)
        else:
            d = self._attach_distal(self._process(idx), sid)
        return self._attach_esm(d, sid)

    def build_cache(self):
        os.makedirs(self.cache_dir, exist_ok=True)
        for i, sid in enumerate(self.ids):
            cache_path = os.path.join(self.cache_dir, f'{sid}.pt')
            if os.path.exists(cache_path):
                continue
            try:
                torch.save(self._process(i), cache_path)
            except Exception:
                continue
            if i % 500 == 0:
                print(f'{i}/{len(self.ids)}')


In [12]:
# [ESSENTIAL] Data augmentation — random SO(3) rotation applied to all coords; AugmentedDataset wraps base dataset for training-time augmentation
import torch
from torch_geometric.data import Data


def random_rotation_matrix(device='cpu'):
    """Sample a uniformly random 3D rotation matrix via QR decomposition."""
    Z = torch.randn(3, 3, device=device)
    Q, R = torch.linalg.qr(Z)
    Q = Q * torch.sign(torch.linalg.det(Q))
    return Q


def augment_data(data: Data) -> Data:
    """Random SO(3) rotation applied to x (14 atoms), pos, pos_distal, y.
    Edge attrs and ESM embeddings are invariant (sequence-based, not coordinate-based)."""
    R = random_rotation_matrix()
    n = data.x.shape[0]
    num_metals = data.num_metals if hasattr(data, 'num_metals') else data.y.shape[0]
    pos_distal = data.pos_distal if hasattr(data, 'pos_distal') else data.pos
    augmented = Data(
        x=(data.x.reshape(n, 14, 3) @ R.T).reshape(n, 42),
        edge_index=data.edge_index,
        edge_attr=data.edge_attr,
        y=data.y @ R.T,
        num_metals=num_metals,
        res_type=data.res_type,
        pos=data.pos @ R.T,
        pos_distal=pos_distal @ R.T,
    )
    # ESM embeddings describe sequence context — they don't rotate with the structure
    if hasattr(data, 'esm'):
        augmented.esm = data.esm
    return augmented


class AugmentedDataset(MetalBindingDataset):
    """Applies a fresh random rotation on every __getitem__ call."""
    def __getitem__(self, idx):
        return augment_data(super().__getitem__(idx))


print('augment_data() and AugmentedDataset ready')


augment_data() and AugmentedDataset ready


In [13]:
# [ESSENTIAL] Model components — MLP helper and MetalMPNNLayer (message-passing layer with sum aggregation)
from torch import nn
from torch_geometric.nn import MessagePassing


class MLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )

    def forward(self, x):
        return self.net(x)


class MetalMPNNLayer(MessagePassing):
    def __init__(self, node_dim, edge_dim, hidden_dim):
        super().__init__(aggr='sum')
        self.message_mlp = MLP(2 * node_dim + edge_dim, hidden_dim)
        self.update_mlp  = MLP(node_dim + hidden_dim, node_dim)

    def forward(self, x, edge_index, edge_attr):
        msg = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        return self.update_mlp(torch.cat([x, msg], dim=-1))

    def message(self, x_i, x_j, edge_attr):
        return self.message_mlp(torch.cat([x_i, x_j, edge_attr], dim=-1))

In [14]:
# [ESSENTIAL] Model components — graph attention mechanisms (softmax/sparsemax/top-k) and MetalPredictionHead
# num_hypotheses=1 → identical to original single-head (self.attention_mlp preserved for checkpoint compat)
# num_hypotheses>1 → winner-takes-all multi-head (self.attention_mlps ModuleList); use wta_loss() in training cell
import torch.nn.functional as F
from torch_geometric.utils import softmax as pyg_softmax


def graph_sparsemax(logits, batch):
    z = logits.squeeze(1)
    num_graphs = batch.max().item() + 1
    out = torch.zeros_like(z)
    for g in range(num_graphs):
        mask = batch == g
        z_g = z[mask]
        n_g = z_g.shape[0]
        z_sorted, _ = torch.sort(z_g, descending=True)
        z_cumsum = torch.cumsum(z_sorted, dim=0)
        k = torch.arange(1, n_g + 1, device=z.device, dtype=z.dtype)
        k_hat = (1 + k * z_sorted > z_cumsum).sum().clamp(min=1)
        tau = (z_cumsum[k_hat - 1] - 1.0) / k_hat
        out[mask] = torch.clamp(z_g - tau, min=0.0)
    return out.unsqueeze(1)


def graph_topk_attn(logits, batch, k):
    z = logits.squeeze(1)
    num_graphs = batch.max().item() + 1
    out = torch.zeros_like(z)
    for g in range(num_graphs):
        mask = batch == g
        z_g = z[mask]
        k_g = min(k, z_g.shape[0])
        top_vals, top_idx = torch.topk(z_g, k_g)
        vals = torch.zeros_like(z_g)
        vals[top_idx] = torch.softmax(top_vals, dim=0)
        out[mask] = vals
    return out.unsqueeze(1)


class MetalPredictionHead(nn.Module):
    """
    attn_mode        : 'softmax' | 'sparsemax' | 'topk'
    top_k            : nodes kept per graph when attn_mode='topk'
    res_emb_dim      : bypass residue embedding concatenated to attention MLP input (0 = off)
    temperature      : initial τ value; logits are divided by τ before softmax (τ < 1 → sharper)
    learn_temperature: if True, τ is a learned nn.Parameter; otherwise fixed buffer
    use_plm          : if True, expects an esm=(N, esm_dim) tensor and uses it instead of res_bypass
    esm_dim          : embedding dimension of the ESM-2 model (480 for esm2_t12_35M_UR50D)
    num_hypotheses   : 1 = standard single head (self.attention_mlp, checkpoint-compatible with all v8 runs)
                       >1 = K parallel heads (self.attention_mlps ModuleList); train with wta_loss()

    Centroid is computed from CA positions (pos); aggregation offset is computed
    from the distal sidechain atom positions (pos_distal). Falls back to CA if
    the sidecar cache hasn't been built yet.
    """
    def __init__(self, node_dim, attn_mode='softmax', top_k=10, res_emb_dim=16,
                 temperature=1.0, learn_temperature=False, use_plm=False, esm_dim=480,
                 num_hypotheses=1):
        super().__init__()
        self.attn_mode      = attn_mode
        self.top_k          = top_k
        self.res_emb_dim    = res_emb_dim
        self.use_plm        = use_plm
        self.num_hypotheses = num_hypotheses

        log_tau = torch.tensor(float(temperature)).log()
        if learn_temperature:
            self.log_temperature = nn.Parameter(log_tau)
        else:
            self.register_buffer('log_temperature', log_tau)

        if use_plm:
            # Project ESM-2 480-dim → node_dim before concat; keeps MLP the same size as baseline
            self.esm_proj = nn.Linear(esm_dim, node_dim)
            attn_in_dim = node_dim + node_dim          # 64 + 64 = 128
        else:
            # Original: small learned embedding keyed by residue type (21 types)
            if res_emb_dim > 0:
                self.res_bypass = nn.Embedding(len(RESIDUES), res_emb_dim)
            attn_in_dim = node_dim + res_emb_dim

        def _make_mlp():
            return nn.Sequential(
                nn.Linear(attn_in_dim, attn_in_dim // 2),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(attn_in_dim // 2, 1)
            )

        if num_hypotheses == 1:
            self.attention_mlp  = _make_mlp()  # single head — name kept for v8 checkpoint compat
        else:
            self.attention_mlps = nn.ModuleList([_make_mlp() for _ in range(num_hypotheses)])

    def _apply_attn(self, logits, batch):
        tau = self.log_temperature.exp()
        logits = logits / tau
        if self.attn_mode == 'softmax':
            return pyg_softmax(logits, batch, dim=0)
        elif self.attn_mode == 'sparsemax':
            return graph_sparsemax(logits, batch)
        elif self.attn_mode == 'topk':
            return graph_topk_attn(logits, batch, self.top_k)
        else:
            raise ValueError(f'Unknown attn_mode: {self.attn_mode!r}')

    def _predict_one(self, mlp, attn_input, centroid, centered_distal, batch):
        """Single MLP → attention weights → weighted offset → (N_graphs, 3) prediction."""
        weights = self._apply_attn(mlp(attn_input), batch)
        num_graphs = centroid.shape[0]
        offset = torch.zeros(num_graphs, 3, device=centroid.device)
        offset.scatter_add_(0, batch.unsqueeze(1).expand(-1, 3), weights * centered_distal)
        return centroid + offset, weights

    def forward(self, x, pos, pos_distal, batch, res_type, return_attn=False, esm=None):
        num_graphs = batch.max().item() + 1

        # MPS-safe counts
        counts = torch.zeros(num_graphs, dtype=torch.float, device=x.device) \
                      .scatter_add(0, batch, torch.ones(batch.shape[0], dtype=torch.float, device=x.device))

        # Centroid from CA (always defined)
        centroid = torch.zeros(num_graphs, 3, device=x.device)
        centroid.scatter_add_(0, batch.unsqueeze(1).expand(-1, 3), pos)
        centroid /= counts.unsqueeze(1)

        # Offset is weighted sum of (distal_atom - centroid_CA)
        centered_distal = pos_distal - centroid[batch]

        if self.use_plm:
            if esm is None:
                raise RuntimeError(
                    'use_plm=True but no ESM embeddings on this batch. '
                    'Run the ESM-2 cache builder cell first, then set '
                    'esm_cache_dir=ESM_CACHE_DIR when constructing the dataset.'
                )
            attn_input = torch.cat([x, self.esm_proj(esm)], dim=-1)
        elif self.res_emb_dim > 0:
            # Original path: learned per-residue-type embedding
            attn_input = torch.cat([x, self.res_bypass(res_type)], dim=-1)
        else:
            attn_input = x

        if self.num_hypotheses == 1:
            pred, weights = self._predict_one(self.attention_mlp, attn_input,
                                              centroid, centered_distal, batch)
            if return_attn:
                return pred, weights.squeeze(1)
            return pred

        # ── Multi-hypothesis: K parallel heads → (N_graphs, K, 3) ───────────────
        preds, all_weights = [], []
        for mlp in self.attention_mlps:
            pred_k, w_k = self._predict_one(mlp, attn_input, centroid, centered_distal, batch)
            preds.append(pred_k)
            all_weights.append(w_k.squeeze(1))
        stacked = torch.stack(preds, dim=1)  # (N, K, 3)
        if return_attn:
            return stacked, all_weights       # list of K (N,) max-weight tensors
        return stacked

In [15]:
# [ESSENTIAL] Full MetalPredictionModel definition (3-layer MPNN + attention head), cache build for pt_cache_v3, and forward-pass sanity check
import torch


class MetalPredictionModel(nn.Module):
    def __init__(self, input_dim=42, node_dim=64, edge_dim=4, hidden_dim=64,
                 attn_mode='softmax', top_k=10, res_emb_dim=16,
                 temperature=1.0, learn_temperature=False,
                 use_plm=False, esm_dim=480, num_hypotheses=1):
        super().__init__()
        self.use_plm       = use_plm
        self.res_embedding = nn.Embedding(len(RESIDUES), node_dim)
        self.node_embed    = nn.Linear(input_dim, node_dim)
        self.layers = nn.ModuleList([
            MetalMPNNLayer(node_dim, edge_dim, hidden_dim)
            for _ in range(3)
        ])
        self.head = MetalPredictionHead(node_dim, attn_mode=attn_mode,
                                        top_k=top_k, res_emb_dim=res_emb_dim,
                                        temperature=temperature,
                                        learn_temperature=learn_temperature,
                                        use_plm=use_plm, esm_dim=esm_dim,
                                        num_hypotheses=num_hypotheses)

    def forward(self, x, edge_index, edge_attr, batch, res_type, pos, pos_distal,
                return_attn=False, esm=None):
        x = self.node_embed(x) + self.res_embedding(res_type)
        for layer in self.layers:
            x = layer(x, edge_index, edge_attr)
        return self.head(x, pos, pos_distal, batch, res_type,
                         return_attn=return_attn, esm=esm)


device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

CHUNK_GLOB = 'data/gemmi_atom14/chunk_*.parquet'
CACHE_DIR  = 'data/pt_cache_v3'

all_ids  = train_ids + val_ids + test_ids
cache_ds = MetalBindingDataset(CHUNK_GLOB, all_ids, cache_dir=CACHE_DIR)
cache_ds.build_cache()

# Sanity check
model = MetalPredictionModel(learn_temperature=True).to(device)
sample = cache_ds[0]
out = model(
    sample.x.to(device),
    sample.edge_index.to(device),
    sample.edge_attr.to(device),
    torch.zeros(sample.x.shape[0], dtype=torch.long, device=device),
    sample.res_type.to(device),
    sample.pos.to(device),
    sample.pos_distal.to(device),
)
print('pred_coords:', out.shape)
print('x shape:', sample.x.shape, '  edge_attr shape:', sample.edge_attr.shape)
print('pos_distal available:', hasattr(sample, 'pos_distal'))
print('attn_mode:', model.head.attn_mode)
print('learn_temperature:', isinstance(model.head.log_temperature, nn.Parameter))

pred_coords: torch.Size([1, 3])
x shape: torch.Size([932, 42])   edge_attr shape: torch.Size([9320, 4])
pos_distal available: True
attn_mode: softmax
learn_temperature: True


In [16]:
# [ESSENTIAL] Filter to single-metal structures and build cluster-aware sm_train/val/test_ids splits
import torch, os, numpy as np

# Filter to single-metal structures
single_metal_ids = []
for sid in all_ids:
    cache_path = f'{CACHE_DIR}/{sid}.pt'
    if not os.path.exists(cache_path):
        continue
    d = torch.load(cache_path, weights_only=False)
    if d.y.shape[0] == 1:
        single_metal_ids.append(sid)

print(f'{len(single_metal_ids)} single-metal structures')

# Redo cluster-aware split using only single-metal structures
single_metal_set = set(single_metal_ids)
sm_cluster_to_structs = {
    c: [s for s in structs if s in single_metal_set]
    for c, structs in cluster_to_structures.items()
}
sm_cluster_to_structs = {c: s for c, s in sm_cluster_to_structs.items() if s}
sm_clusters = list(sm_cluster_to_structs.keys())

rng = np.random.default_rng(42)
rng.shuffle(sm_clusters)
n = len(sm_clusters)

sm_train_ids = [s for c in sm_clusters[:int(0.8*n)]          for s in sm_cluster_to_structs[c]]
sm_val_ids   = [s for c in sm_clusters[int(0.8*n):int(0.9*n)] for s in sm_cluster_to_structs[c]]
sm_test_ids  = [s for c in sm_clusters[int(0.9*n):]           for s in sm_cluster_to_structs[c]]

print(f'train: {len(sm_train_ids)}  val: {len(sm_val_ids)}  test: {len(sm_test_ids)}')
#time to execute:

12726 single-metal structures
train: 10477  val: 1043  test: 1206


In [17]:
# [ESSENTIAL] Build cluster-aware all-metal splits (am_train/val/test_ids) for multi-metal training mode
import os, torch, numpy as np

# Cluster-aware split: all structures with >= 1 metal
valid_set = set()
for sid in all_ids:
    cache_path = f'{CACHE_DIR}/{sid}.pt'
    if not os.path.exists(cache_path):
        continue
    d = torch.load(cache_path, weights_only=False)
    if d.y.shape[0] >= 1:
        valid_set.add(sid)

am_cluster_to_structs = {
    c: [s for s in structs if s in valid_set]
    for c, structs in cluster_to_structures.items()
}
am_cluster_to_structs = {c: s for c, s in am_cluster_to_structs.items() if s}
am_clusters = list(am_cluster_to_structs.keys())

rng = np.random.default_rng(42)
rng.shuffle(am_clusters)
n = len(am_clusters)

am_train_ids = [s for c in am_clusters[:int(0.8*n)]           for s in am_cluster_to_structs[c]]
am_val_ids   = [s for c in am_clusters[int(0.8*n):int(0.9*n)] for s in am_cluster_to_structs[c]]
am_test_ids  = [s for c in am_clusters[int(0.9*n):]           for s in am_cluster_to_structs[c]]

print(f'All-metal split:')
print(f'  train: {len(am_train_ids)}  val: {len(am_val_ids)}  test: {len(am_test_ids)}')
print(f'  (vs single-metal: {len(sm_train_ids)} / {len(sm_val_ids)} / {len(sm_test_ids)})')


All-metal split:
  train: 28881  val: 3717  test: 3134
  (vs single-metal: 10477 / 1043 / 1206)


In [18]:
# [ESSENTIAL] Metal res_name filter — bare-metal-only and zinc-only structure ID lists
# Reads parquet res_name to exclude organometallics (HEM, FES, SF4, etc.) without cache rebuild.
# Note: heme Fe does coordinate axially to His/Met, but the Fe sits inside a porphyrin ring —
# its protein-coordination geometry is fundamentally different from bare metal ions.
import glob, pandas as pd

# Bare metal ion res_names (excludes hemes, iron-sulfur clusters, porphyrin-bound metals, etc.)
BARE_METAL_RESNAMES = {
    'ZN', 'MG', 'FE', 'CA', 'CU', 'MN', 'CO', 'NI', 'MO',
    'FE2', 'CD', 'V', 'W',
}

def build_metal_resname_index(chunk_glob):
    """Returns {structure_id: set(metal_res_names)} by scanning parquet files."""
    index = {}
    for fpath in glob.glob(chunk_glob):
        df = pd.read_parquet(fpath, columns=['structure_id', 'res_name', 'is_metal'])
        metal_df = df[df['is_metal']]
        for sid, grp in metal_df.groupby('structure_id'):
            index[str(sid)] = set(grp['res_name'].str.strip().str.upper())
    return index

def filter_by_metal_resnames(ids, metal_index, allowed):
    """Keep only structures whose every metal res_name is in allowed."""
    allowed_upper = {r.upper() for r in allowed}
    return [sid for sid in ids
            if sid in metal_index and metal_index[sid].issubset(allowed_upper)]

# Scan parquets once (~30 s); builds a dict keyed by structure_id
print('Scanning parquets for metal res_names...')
metal_resname_index = build_metal_resname_index(CHUNK_GLOB)

# Bare-metal-only single-metal splits (no HEM, FES, SF4, CHL, etc.)
bm_train_ids = filter_by_metal_resnames(sm_train_ids, metal_resname_index, BARE_METAL_RESNAMES)
bm_val_ids   = filter_by_metal_resnames(sm_val_ids,   metal_resname_index, BARE_METAL_RESNAMES)
bm_test_ids  = filter_by_metal_resnames(sm_test_ids,  metal_resname_index, BARE_METAL_RESNAMES)

# Zinc-only single-metal splits
zn_train_ids = filter_by_metal_resnames(sm_train_ids, metal_resname_index, {'ZN'})
zn_val_ids   = filter_by_metal_resnames(sm_val_ids,   metal_resname_index, {'ZN'})
zn_test_ids  = filter_by_metal_resnames(sm_test_ids,  metal_resname_index, {'ZN'})

print(f'Bare-metals single-metal: train {len(bm_train_ids)} | val {len(bm_val_ids)} | test {len(bm_test_ids)}')
print(f'Zinc-only   single-metal: train {len(zn_train_ids)} | val {len(zn_val_ids)} | test {len(zn_test_ids)}')
print(f'(Unfiltered single-metal: train {len(sm_train_ids)} | val {len(sm_val_ids)} | test {len(sm_test_ids)})')

Scanning parquets for metal res_names...
Bare-metals single-metal: train 8039 | val 873 | test 986
Zinc-only   single-metal: train 4285 | val 146 | test 297
(Unfiltered single-metal: train 10477 | val 1043 | test 1206)


In [19]:
# [ESSENTIAL] Run logging — appends/updates a row in data/runs.csv after training or test eval
# Call log_run('train', ...) at the end of the training cell and log_run('test', ...) at the end of the test cell.
import os, pandas as pd
from datetime import datetime

RUNS_CSV = 'data/runs.csv'

def log_run(phase, checkpoint,
            # training params
            attn_mode=None, metal_filter=None, num_hypotheses=None,
            diversity_weight=None, learn_temperature=None, use_plm=None,
            batch_size=None, num_epochs=None, lr=None, grad_clip=None, k=None,
            # train-phase metrics
            train_size=None, val_size=None,
            best_epoch=None, best_train_loss=None, best_val_rmse=None,
            # test-phase metrics
            test_size=None, test_rmse=None, test_mean=None, test_median=None,
            test_pct_2a=None, test_pct_5a=None):
    """
    Append or update a row in data/runs.csv keyed by checkpoint filename.
    Call with phase='train' at end of training loop, phase='test' at end of test eval.
    The two calls merge into a single row so all columns are populated after testing.
    """
    row = dict(
        checkpoint=os.path.basename(checkpoint),
        attn_mode=attn_mode, metal_filter=metal_filter,
        num_hypotheses=num_hypotheses, diversity_weight=diversity_weight,
        learn_temperature=learn_temperature, use_plm=use_plm,
        batch_size=batch_size, num_epochs=num_epochs, lr=lr,
        grad_clip=grad_clip, k=k,
        train_size=train_size, val_size=val_size,
        best_epoch=best_epoch, best_train_loss=best_train_loss,
        best_val_rmse=best_val_rmse,
        test_size=test_size, test_rmse=test_rmse,
        test_mean=test_mean, test_median=test_median,
        test_pct_2a=test_pct_2a, test_pct_5a=test_pct_5a,
        last_updated=datetime.now().strftime('%Y-%m-%d %H:%M'),
    )

    if os.path.exists(RUNS_CSV):
        df = pd.read_csv(RUNS_CSV)
        mask = df['checkpoint'] == row['checkpoint']
        if mask.any():
            # Update existing row: overwrite only non-None fields
            idx = df.index[mask][0]
            for col, val in row.items():
                if val is not None:
                    df.at[idx, col] = val
        else:
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])

    df.to_csv(RUNS_CSV, index=False)
    print(f'[log_run] {phase} metrics saved → {RUNS_CSV}  (checkpoint: {row["checkpoint"]})')

print('log_run() ready — will write to', RUNS_CSV)

log_run() ready — will write to data/runs.csv


In [24]:
# [ESSENTIAL] Training loop — config, dataloader setup, model/optimizer/scheduler init, epoch loop with val RMSE, LR scheduling, and checkpointing
import time
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

# ── Config ────────────────────────────────────────────────────────────────────
SINGLE_METAL_ONLY = True      # False → train on all metals (am_ splits)
ATTN_MODE         = 'softmax' # 'softmax' | 'sparsemax' | 'topk'
RESUME            = False     # True → load checkpoint and continue training
LEARN_TEMPERATURE = True
TEMPERATURE       = 1.0
USE_PLM           = False     # True → inject frozen ESM-2 embeddings into the attention head
ESM_CACHE_DIR     = 'data/esm2_cache'   # built by the ESM-2 cache builder cell (non-essential section)
ESM_DIM           = 480                 # esm2_t12_35M_UR50D output dimension

# ── Metal filter ──────────────────────────────────────────────────────────────
# None    → no filter, use full single/all-metal split (original behavior)
# 'bare'  → bare metal ions only (excludes HEM, FES, SF4, etc.); requires metal filter cell
# 'ZN'    → zinc-only; requires metal filter cell
METAL_FILTER = 'ZN'

# ── Multi-hypothesis (winner-takes-all) ───────────────────────────────────────
# 1       → standard single-head prediction (original behavior, v8-compatible checkpoints)
# >1      → K parallel attention heads; training uses wta_loss(); val/test scores best hypothesis
NUM_HYPOTHESES   = 3
DIVERSITY_WEIGHT = 0.05   # weight of diversity penalty when NUM_HYPOTHESES > 1

BATCH_SIZE  = 16
NUM_EPOCHS  = 25
LR          = 1e-3
GRAD_CLIP   = 1.0
K_NEIGHBORS = 16

# ── Split selection ───────────────────────────────────────────────────────────
if METAL_FILTER == 'ZN':
    train_ids_use, val_ids_use = zn_train_ids, zn_val_ids
    filter_tag = '_zn'
elif METAL_FILTER == 'bare':
    train_ids_use, val_ids_use = bm_train_ids, bm_val_ids
    filter_tag = '_bm'
elif SINGLE_METAL_ONLY:
    train_ids_use, val_ids_use = sm_train_ids, sm_val_ids
    filter_tag = '_sm'
else:
    train_ids_use, val_ids_use = am_train_ids, am_val_ids
    filter_tag = ''

plm_tag = '_plm' if USE_PLM else ''
wta_tag = f'_wta{NUM_HYPOTHESES}' if NUM_HYPOTHESES > 1 else ''
CHECKPOINT = f'data/best_model_v9{filter_tag}_{ATTN_MODE}{plm_tag}{wta_tag}.pt'
CKPT_FULL  = f'data/best_model_v9{filter_tag}_{ATTN_MODE}{plm_tag}{wta_tag}_full.pt'

esm_dir = ESM_CACHE_DIR if USE_PLM else None

print(f'Filter: {METAL_FILTER or ("single-metal" if SINGLE_METAL_ONLY else "all-metal")} | '
      f'attn: {ATTN_MODE} | hypotheses: {NUM_HYPOTHESES} | '
      f'resume: {RESUME} | learn_temperature: {LEARN_TEMPERATURE} | use_plm: {USE_PLM}')
print(f'train: {len(train_ids_use)}  val: {len(val_ids_use)}')
print(f'checkpoint: {CHECKPOINT}')


# ── Winner-takes-all loss ─────────────────────────────────────────────────────
def wta_loss(preds, targets, diversity_weight=0.05):
    """
    preds  : (N, K, 3) — K hypotheses per structure
    targets: (N, 3)    — ground-truth metal position (single-metal mode)
    Returns WTA MSE (best hypothesis only) + diversity regularization that
    pushes heads apart, preventing mode collapse.
    """
    sq_dists = ((preds - targets.unsqueeze(1)) ** 2).sum(-1)  # (N, K)
    wta = sq_dists.min(dim=1).values.mean()
    K = preds.shape[1]
    div = sum(
        torch.exp(-(preds[:, i] - preds[:, j]).norm(dim=-1)).mean()
        for i in range(K) for j in range(i + 1, K)
    ) / (K * (K - 1) / 2)
    return wta + diversity_weight * div


# ── Dataset / DataLoader ──────────────────────────────────────────────────────
train_ds = AugmentedDataset(CHUNK_GLOB, train_ids_use, cache_dir=CACHE_DIR, esm_cache_dir=esm_dir, k=K_NEIGHBORS)
val_ds   = MetalBindingDataset(CHUNK_GLOB, val_ids_use,   cache_dir=CACHE_DIR, esm_cache_dir=esm_dir, k=K_NEIGHBORS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ── Model / Optimizer / Scheduler ────────────────────────────────────────────
model     = MetalPredictionModel(attn_mode=ATTN_MODE,
                                 learn_temperature=LEARN_TEMPERATURE,
                                 temperature=TEMPERATURE,
                                 use_plm=USE_PLM,
                                 esm_dim=ESM_DIM,
                                 num_hypotheses=NUM_HYPOTHESES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, min_lr=1e-6)

start_epoch          = 1
best_val_loss        = float('inf')
best_epoch           = 0
best_train_loss_snap = float('inf')

if RESUME and os.path.exists(CKPT_FULL):
    ckpt = torch.load(CKPT_FULL, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch          = ckpt['epoch'] + 1
    best_val_loss        = ckpt['best_val_loss']
    best_epoch           = ckpt.get('best_epoch', ckpt['epoch'])
    best_train_loss_snap = ckpt.get('best_train_loss_snap', float('inf'))
    print(f'Resumed from epoch {ckpt["epoch"]} | best val RMSE: {best_val_loss:.2f} Å')
elif RESUME and os.path.exists(CHECKPOINT):
    model.load_state_dict(torch.load(CHECKPOINT, weights_only=True))
    print(f'Loaded weights from {CHECKPOINT} (no optimizer state)')
else:
    # Fresh start: zero-init final attention layer(s) → offset=0 at epoch 1 (centroid baseline)
    if NUM_HYPOTHESES > 1:
        for mlp in model.head.attention_mlps:
            nn.init.zeros_(mlp[-1].weight)
            nn.init.zeros_(mlp[-1].bias)
    else:
        nn.init.zeros_(model.head.attention_mlp[-1].weight)
        nn.init.zeros_(model.head.attention_mlp[-1].bias)
    print('Fresh start with zero-init attention head(s)')

# ── Epoch loop ────────────────────────────────────────────────────────────────
for epoch in range(start_epoch, NUM_EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    t0 = time.time()
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        esm_feat = batch.esm.to(device) if (USE_PLM and hasattr(batch, 'esm')) else None
        pred_out = model(batch.x, batch.edge_index, batch.edge_attr,
                         batch.batch, batch.res_type, batch.pos, batch.pos_distal,
                         esm=esm_feat)

        if SINGLE_METAL_ONLY or METAL_FILTER is not None:
            targets = batch.y
        else:
            y_split = torch.split(batch.y, batch.num_metals.tolist())
            targets = torch.stack([
                metals[torch.randint(len(metals), (1,)).item()]
                for metals in y_split
            ])

        if NUM_HYPOTHESES > 1:
            loss = wta_loss(pred_out, targets, DIVERSITY_WEIGHT)
        else:
            loss = F.mse_loss(pred_out, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    val_sq_dists = []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            esm_feat = batch.esm.to(device) if (USE_PLM and hasattr(batch, 'esm')) else None
            pred_out = model(batch.x, batch.edge_index, batch.edge_attr,
                             batch.batch, batch.res_type, batch.pos, batch.pos_distal,
                             esm=esm_feat)

            if NUM_HYPOTHESES > 1:
                sq = ((pred_out - batch.y.unsqueeze(1)) ** 2).sum(-1)
                dists = sq.min(dim=1).values.sqrt()
            elif SINGLE_METAL_ONLY or METAL_FILTER is not None:
                dists = torch.norm(pred_out - batch.y, dim=-1)
            else:
                y_split = torch.split(batch.y, batch.num_metals.tolist())
                dists = torch.stack([
                    torch.norm(pred_out[i] - metals, dim=-1).min()
                    for i, metals in enumerate(y_split)
                ])

            val_sq_dists.append(dists.pow(2))

    val_rmse = torch.cat(val_sq_dists).mean().sqrt().item()
    scheduler.step(val_rmse)

    tau_str = ''
    if LEARN_TEMPERATURE:
        tau = model.head.log_temperature.exp().item()
        tau_str = f' | τ={tau:.3f}'

    flag = ''
    if val_rmse < best_val_loss:
        best_val_loss        = val_rmse
        best_epoch           = epoch
        best_train_loss_snap = train_loss
        torch.save(model.state_dict(), CHECKPOINT)
        torch.save({
            'epoch':               epoch,
            'model':               model.state_dict(),
            'optimizer':           optimizer.state_dict(),
            'scheduler':           scheduler.state_dict(),
            'best_val_loss':       best_val_loss,
            'best_epoch':          best_epoch,
            'best_train_loss_snap': best_train_loss_snap,
        }, CKPT_FULL)
        flag = '  *'

    print(f'Epoch {epoch:3d} | train_loss {train_loss:.4f} | val_rmse {val_rmse:.2f} Å'
          f'{tau_str} | {time.time()-t0:.1f}s{flag}')

# ── Log training result ───────────────────────────────────────────────────────
log_run('train', CHECKPOINT,
        attn_mode=ATTN_MODE, metal_filter=METAL_FILTER,
        num_hypotheses=NUM_HYPOTHESES, diversity_weight=DIVERSITY_WEIGHT if NUM_HYPOTHESES > 1 else None,
        learn_temperature=LEARN_TEMPERATURE, use_plm=USE_PLM,
        batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS, lr=LR,
        grad_clip=GRAD_CLIP, k=K_NEIGHBORS,
        train_size=len(train_ids_use), val_size=len(val_ids_use),
        best_epoch=best_epoch,
        best_train_loss=round(best_train_loss_snap, 4),
        best_val_rmse=round(best_val_loss, 4))

Filter: ZN | attn: softmax | hypotheses: 3 | resume: False | learn_temperature: True | use_plm: False
train: 4285  val: 146
checkpoint: data/best_model_v9_zn_softmax_wta3.pt
Fresh start with zero-init attention head(s)
Epoch   1 | train_loss 178.4998 | val_rmse 14.39 Å | τ=1.030 | 163.6s  *
Epoch   2 | train_loss 179.1993 | val_rmse 14.48 Å | τ=1.086 | 185.6s
Epoch   3 | train_loss 168.3931 | val_rmse 20.00 Å | τ=1.112 | 150.4s
Epoch   4 | train_loss 171.0981 | val_rmse 18.77 Å | τ=0.995 | 187.0s
Epoch   5 | train_loss 143.2881 | val_rmse 13.32 Å | τ=0.972 | 135.4s  *
Epoch   6 | train_loss 112.5875 | val_rmse 12.42 Å | τ=0.930 | 143.8s  *
Epoch   7 | train_loss 96.1571 | val_rmse 12.35 Å | τ=0.879 | 147.9s  *
Epoch   8 | train_loss 72.6693 | val_rmse 11.95 Å | τ=0.853 | 147.8s  *
Epoch   9 | train_loss 72.3602 | val_rmse 12.30 Å | τ=0.832 | 126.4s
Epoch  10 | train_loss 70.2322 | val_rmse 12.05 Å | τ=0.818 | 150.4s
Epoch  11 | train_loss 67.1838 | val_rmse 11.73 Å | τ=0.801 | 134.4s  

In [21]:
# [ESSENTIAL] Test set evaluation — loads best checkpoint and computes RMSE, mean/median error, and accuracy at 2 Å / 5 Å thresholds
import torch
from torch_geometric.loader import DataLoader

# Picks up SINGLE_METAL_ONLY, ATTN_MODE, USE_PLM, ESM_CACHE_DIR, ESM_DIM,
# METAL_FILTER, NUM_HYPOTHESES, CHECKPOINT from the training cell
if METAL_FILTER == 'ZN':
    test_ids_use = zn_test_ids
elif METAL_FILTER == 'bare':
    test_ids_use = bm_test_ids
elif SINGLE_METAL_ONLY:
    test_ids_use = sm_test_ids
else:
    test_ids_use = am_test_ids

esm_dir   = ESM_CACHE_DIR if USE_PLM else None
test_ds   = MetalBindingDataset(CHUNK_GLOB, test_ids_use, cache_dir=CACHE_DIR, esm_cache_dir=esm_dir)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = MetalPredictionModel(attn_mode=ATTN_MODE, use_plm=USE_PLM, esm_dim=ESM_DIM,
                              num_hypotheses=NUM_HYPOTHESES).to(device)
model.load_state_dict(torch.load(CHECKPOINT, weights_only=True))
model.eval()

errors = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        esm_feat = batch.esm.to(device) if (USE_PLM and hasattr(batch, 'esm')) else None
        pred_out = model(batch.x, batch.edge_index, batch.edge_attr,
                         batch.batch, batch.res_type, batch.pos, batch.pos_distal,
                         esm=esm_feat)

        if NUM_HYPOTHESES > 1:
            sq = ((pred_out - batch.y.unsqueeze(1)) ** 2).sum(-1)
            dists = sq.min(dim=1).values.sqrt()
        elif SINGLE_METAL_ONLY or METAL_FILTER is not None:
            dists = torch.norm(pred_out - batch.y, dim=-1)
        else:
            y_split = torch.split(batch.y, batch.num_metals.tolist())
            dists = torch.stack([
                torch.norm(pred_out[i] - metals, dim=-1).min()
                for i, metals in enumerate(y_split)
            ])

        errors.extend(dists.cpu().tolist())

errors = torch.tensor(errors)
t_rmse   = errors.pow(2).mean().sqrt().item()
t_mean   = errors.mean().item()
t_median = errors.median().item()
t_pct_2a = (errors < 2).float().mean().item() * 100
t_pct_5a = (errors < 5).float().mean().item() * 100

print(f'Checkpoint:   {CHECKPOINT}')
print(f'Test set:     {len(test_ids_use)} structures  |  filter: {METAL_FILTER}  |  hypotheses: {NUM_HYPOTHESES}')
print(f'Test RMSE:    {t_rmse:.2f} Å')
print(f'Mean error:   {t_mean:.2f} Å')
print(f'Median error: {t_median:.2f} Å')
print(f'Error < 2 Å:  {t_pct_2a:.1f}%')
print(f'Error < 5 Å:  {t_pct_5a:.1f}%')

# ── Log test result ───────────────────────────────────────────────────────────
log_run('test', CHECKPOINT,
        test_size=len(test_ids_use),
        test_rmse=round(t_rmse, 4), test_mean=round(t_mean, 4),
        test_median=round(t_median, 4),
        test_pct_2a=round(t_pct_2a, 2), test_pct_5a=round(t_pct_5a, 2))

Checkpoint:   data/best_model_v9_zn_softmax_wta3.pt
Test set:     297 structures  |  filter: ZN  |  hypotheses: 3
Test RMSE:    12.31 Å
Mean error:   9.60 Å
Median error: 5.38 Å
Error < 2 Å:  7.4%
Error < 5 Å:  33.7%
[log_run] test metrics saved → data/runs.csv  (checkpoint: best_model_v9_zn_softmax_wta3.pt)


In [ ]:
from metal_predictor.inference import neighborhood_predict

print('neighborhood_predict() ready (from metal_predictor.inference)')

In [23]:
# [NON-ESSENTIAL] Compare soft-attention vs. neighborhood prediction on the test set
# Requires: training cell has been run (model, test_loader, CHECKPOINT, USE_PLM, device in scope)
import torch

model.eval()
soft_errors = []
neigh_errors = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        esm_feat = batch.esm.to(device) if (USE_PLM and hasattr(batch, 'esm')) else None

        # Soft-attention: weighted sum over all residues (current head output)
        soft_pred = model(batch.x, batch.edge_index, batch.edge_attr,
                          batch.batch, batch.res_type, batch.pos, batch.pos_distal,
                          esm=esm_feat)

        # Neighborhood: CA centroid of the top-scoring local cluster
        neigh_pred = neighborhood_predict(model, batch, use_plm=USE_PLM, device=device)

        if SINGLE_METAL_ONLY:
            targets = batch.y
            soft_errors.extend(torch.norm(soft_pred  - targets, dim=-1).cpu().tolist())
            neigh_errors.extend(torch.norm(neigh_pred - targets, dim=-1).cpu().tolist())
        else:
            y_split = torch.split(batch.y, batch.num_metals.tolist())
            for i, metals in enumerate(y_split):
                soft_errors.append(torch.norm(soft_pred[i]  - metals, dim=-1).min().item())
                neigh_errors.append(torch.norm(neigh_pred[i] - metals, dim=-1).min().item())

soft_errors  = torch.tensor(soft_errors)
neigh_errors = torch.tensor(neigh_errors)

print(f'Checkpoint: {CHECKPOINT}  |  use_plm: {USE_PLM}')
print()
print(f'{"Method":<22} {"RMSE":>8} {"Mean":>8} {"Median":>8} {"<2Å":>8} {"<5Å":>8}')
print('-' * 62)
for name, errs in [('Soft-attention', soft_errors), ('Neighborhood', neigh_errors)]:
    print(f'{name:<22} '
          f'{errs.pow(2).mean().sqrt():>7.2f}Å '
          f'{errs.mean():>7.2f}Å '
          f'{errs.median():>7.2f}Å '
          f'{(errs < 2).float().mean()*100:>7.1f}% '
          f'{(errs < 5).float().mean()*100:>7.1f}%')

# Spot-check: top-5 residues by neighborhood score for the first structure in the last batch
batch0 = next(iter(test_loader)).to(device)
esm0   = batch0.esm.to(device) if (USE_PLM and hasattr(batch0, 'esm')) else None
_, attn0 = model(batch0.x, batch0.edge_index, batch0.edge_attr,
                 batch0.batch, batch0.res_type, batch0.pos, batch0.pos_distal,
                 return_attn=True, esm=esm0)
neigh_score0 = attn0.clone()
neigh_score0.scatter_add_(0, batch0.edge_index[0], attn0[batch0.edge_index[1]])

mask0    = batch0.batch == 0
node_idx = mask0.nonzero(as_tuple=True)[0]
scores_g = neigh_score0[mask0]
top5_local = scores_g.topk(5).indices

print()
print('Top-5 residues by neighborhood score (first test structure):')
print(f'  {"Rank":<6} {"ResType":<10} {"NeighScore":>12} {"AttnWeight":>12}')
for rank, li in enumerate(top5_local, 1):
    ni  = node_idx[li].item()
    rt  = batch0.res_type[ni].item()
    print(f'  {rank:<6} {RESIDUES[rt]:<10} {scores_g[li].item():>12.4f} {attn0[ni].item():>12.4f}')


RuntimeError: The size of tensor a (3) must match the size of tensor b (16) at non-singleton dimension 1

# Non-Essential Cells

The cells below are not part of the core model pipeline. They include one-time setup scripts, sanity checks, exploratory analysis, and visualizations.

In [ ]:
# [NON-ESSENTIAL] Redundant numpy/pandas imports — already imported in subsequent cells

import numpy as np
import pandas as pd

In [ ]:
# [NON-ESSENTIAL] Sanity check — read one structure from parquet to verify data access
pqpath = 'data/clusters/clusters.parquet'
sid = train_ids[0]
print(type(sid), sid)
df = pd.read_parquet(pqpath, filters=[('structure_id', '==', sid)])
print(df.shape)

In [ ]:
# [NON-ESSENTIAL] Old dataset instantiation (uses outdated cache dir data/pt_cache); superseded by cell using data/pt_cache_v3
CHUNK_GLOB = 'data/gemmi_atom14/chunk_*.parquet'
CACHE_DIR  = 'data/pt_cache'

train_ds = MetalBindingDataset(CHUNK_GLOB, train_ids, k=25, cache_dir=CACHE_DIR)
val_ds   = MetalBindingDataset(CHUNK_GLOB, val_ids,   k=25, cache_dir=CACHE_DIR)
test_ds  = MetalBindingDataset(CHUNK_GLOB, test_ids,  k=25, cache_dir=CACHE_DIR)

print(f'train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}')


In [ ]:
# [NON-ESSENTIAL] Sanity check — print shape info for one training sample

# Sanity check one sample
sample = train_ds[0]
print(sample)
print('x:', sample.x.shape, 'edge_index:', sample.edge_index.shape,
      'edge_attr:', sample.edge_attr.shape, 'y:', sample.y.shape)

In [ ]:
# [NON-ESSENTIAL] One-time cache build using old cache dir; run once to pre-process data, idempotent if interrupted
# Build cache for all splits — run once overnight (~14 hrs), idempotent if interrupted
all_ids = train_ids + val_ids + test_ids
cache_ds = MetalBindingDataset(CHUNK_GLOB, all_ids, k=25, cache_dir=CACHE_DIR)
cache_ds.build_cache()

In [ ]:
# [NON-ESSENTIAL] One-time data patch — adds Ca/Cd/V/W metal coordinates missing from the initial parquet parse; run once after build_cache()
# ── Patch: re-add Ca/Cd/V/W metals missing from parquet parse ────────────────
# The original gemmi_atom14 parquets omitted these elements. Run this ONCE
# after build_cache() completes to patch zero-metal structures in pt_cache_v3.
# Then re-run cell 06a457cf to rebuild sm_train_ids / sm_val_ids / sm_test_ids.

import gemmi, glob, os, torch

CIF_DIR   = 'data/cifs'
ADD_METALS = {'Ca', 'Cd', 'V', 'W'}

zero_metal_ids = []
for f in glob.glob(f'{CACHE_DIR}/*.pt'):
    d = torch.load(f, weights_only=False)
    if d.y.shape[0] == 0:
        zero_metal_ids.append(os.path.basename(f).replace('.pt', ''))

print(f'Zero-metal structures to patch: {len(zero_metal_ids)}')

patched = no_metal = no_cif = errors = 0
found_by = {el: 0 for el in ADD_METALS}

for sid in zero_metal_ids:
    cif_path = os.path.join(CIF_DIR, sid)
    if not os.path.exists(cif_path):
        no_cif += 1
        continue
    try:
        model_gemmi = gemmi.read_structure(cif_path)[0]
        new_coords = []
        for chain in model_gemmi:
            for residue in chain:
                if len(residue) > 0 and residue[0].element.name in ADD_METALS:
                    a = residue[0]
                    new_coords.append([a.pos.x, a.pos.y, a.pos.z])
                    found_by[residue[0].element.name] += 1
        if not new_coords:
            no_metal += 1
            continue
        cache_path = os.path.join(CACHE_DIR, f'{sid}.pt')
        d = torch.load(cache_path, weights_only=False)
        d.y = torch.tensor(new_coords, dtype=torch.float)
        torch.save(d, cache_path)
        patched += 1
    except Exception:
        errors += 1

print(f'Patched: {patched}  |  no target metal: {no_metal}  |  no CIF: {no_cif}  |  errors: {errors}')
print('Metal sites added:', {el: found_by[el] for el in ADD_METALS if found_by[el] > 0})
print('→ Now re-run cell 06a457cf to rebuild sm_train_ids / sm_val_ids / sm_test_ids')


In [ ]:
# [NON-ESSENTIAL] Exploratory analysis — mean CA-to-distal sidechain atom distance per residue type
import torch, os
from collections import defaultdict

IDX_TO_RES = {i: r for r, i in RES_TO_IDX.items()}
dists_by_res = defaultdict(list)

n_checked = 0
for sid in train_ids[:500]:     # sample 500 structures
    cache_path = os.path.join(CACHE_DIR, f'{sid}.pt')
    if not os.path.exists(cache_path):
        continue
    d = torch.load(cache_path, weights_only=False)
    # atom14: (N, 14, 3) — use pos_distal (last real sidechain atom) vs CA
    ca        = d.pos                    # (N, 3) CA positions
    distal    = d.pos_distal             # (N, 3) last-finite atom per residue
    valid     = (distal - ca).norm(dim=-1) > 0.1   # exclude residues where distal == CA
    dists     = (distal - ca).norm(dim=-1)
    for res_idx, dist, v in zip(d.res_type.tolist(), dists.tolist(), valid.tolist()):
        if v:
            dists_by_res[IDX_TO_RES[res_idx]].append(dist)
    n_checked += 1

print(f'Sampled {n_checked} structures\n')
print(f'{"Residue":6s}  {"N":>6s}  {"mean CA→distal":>15s}  {"std":>6s}')
print('─' * 40)
for res in RESIDUES:
    ds = dists_by_res.get(res, [])
    if not ds:
        print(f'{res:6s}  {"—":>6s}  {"—":>15s}  {"—":>6s}')
        continue
    t = torch.tensor(ds)
    print(f'{res:6s}  {len(ds):>6d}  {t.mean():>14.2f} Å  {t.std():>5.2f}')


In [ ]:
# [NON-ESSENTIAL] One-time build of distal atom position sidecar cache (pt_cache_distal); this data is now embedded in pt_cache_v3
import glob, os, shutil, torch
import pandas as pd
import numpy as np

DISTAL_CACHE_DIR = 'data/pt_cache_distal'

# Wipe any stale/partial files from previous interrupted runs
if os.path.exists(DISTAL_CACHE_DIR):
    shutil.rmtree(DISTAL_CACHE_DIR)
    print(f'Cleared stale {DISTAL_CACHE_DIR}')
os.makedirs(DISTAL_CACHE_DIR, exist_ok=True)

coord_cols = [(f'x{i}', f'y{i}', f'z{i}') for i in range(14)]

# Build index: sid → parquet path
index = {}
for fpath in glob.glob(CHUNK_GLOB):
    df_idx = pd.read_parquet(fpath, columns=['structure_id'])
    for sid in df_idx['structure_id'].unique():
        index[sid] = fpath

n_done = n_skip = n_err = 0
for sid in all_ids:
    out_path = os.path.join(DISTAL_CACHE_DIR, f'{sid}.pt')
    if os.path.exists(out_path):
        n_skip += 1
        continue
    if sid not in index:
        n_err += 1
        continue
    try:
        df = pd.read_parquet(index[sid])
        df = df[(df['structure_id'] == sid) & (~df['is_metal'])].reset_index(drop=True)

        ca_ok = np.isfinite(df[['x1','y1','z1']].values).all(axis=1)
        df = df[ca_ok].reset_index(drop=True)

        coords14 = df[[c for triple in coord_cols for c in triple]].values \
                     .reshape(len(df), 14, 3)

        finite_mask = np.isfinite(coords14).all(axis=2)
        last_idx    = (finite_mask * np.arange(14)).argmax(axis=1)
        last_idx[~finite_mask.any(axis=1)] = 1  # CA fallback

        pos_distal = coords14[np.arange(len(df)), last_idx, :]
        torch.save(torch.tensor(pos_distal, dtype=torch.float), out_path)
        n_done += 1
    except Exception as e:
        n_err += 1

    if (n_done + n_skip) % 2000 == 0:
        print(f'done: {n_done}  skipped: {n_skip}  errors: {n_err}')

print(f'Finished. done: {n_done}  skipped: {n_skip}  errors: {n_err}')


In [ ]:
# [NON-ESSENTIAL] Timing probe — estimates wall-clock time to build the full distal atom cache
import time, glob, os, torch
import pandas as pd
import numpy as np

# ── Timing probe: estimate full distal-cache build time ───────────────────────
N_PROBE = 100   # structures to time

coord_cols = [(f'x{i}', f'y{i}', f'z{i}') for i in range(14)]

# Re-use the index built in swltcghl9l (or rebuild a quick one)
probe_index = {}
for fpath in glob.glob(CHUNK_GLOB):
    df_idx = pd.read_parquet(fpath, columns=['structure_id'])
    for sid in df_idx['structure_id'].unique():
        probe_index[sid] = fpath

probe_ids = [sid for sid in all_ids if sid in probe_index][:N_PROBE]

t0 = time.time()
for sid in probe_ids:
    df = pd.read_parquet(probe_index[sid])
    df = df[(df['structure_id'] == sid) & (~df['is_metal'])].reset_index(drop=True)
    ca_ok = np.isfinite(df[['x1','y1','z1']].values).all(axis=1)
    df = df[ca_ok].reset_index(drop=True)
    coords14 = df[[c for triple in coord_cols for c in triple]].values.reshape(len(df), 14, 3)
    finite_mask = np.isfinite(coords14).all(axis=2)
    last_idx = (finite_mask * np.arange(14)).argmax(axis=1)
    last_idx[~finite_mask.any(axis=1)] = 1
    _ = coords14[np.arange(len(df)), last_idx, :]
elapsed = time.time() - t0

per_structure = elapsed / N_PROBE
total_est     = per_structure * len(all_ids)

print(f'Timed {N_PROBE} structures in {elapsed:.1f}s  ({per_structure*1000:.1f} ms/structure)')
print(f'Total structures: {len(all_ids)}')
print(f'Estimated full build: {total_est/60:.1f} min  ({total_est/3600:.2f} hr)')
print()
print('(This is parquet read + atom14 parse only — no disk write)')
write_est = total_est * 1.15   # ~15% overhead for torch.save
print(f'With torch.save overhead (~15%): {write_est/60:.1f} min  ({write_est/3600:.2f} hr)')


In [ ]:
# [NON-ESSENTIAL] Exploratory analysis — distribution of metal site counts (0, 1, 2, ...) across all cached structures
from collections import Counter

metal_counts = Counter()
for sid in all_ids:
    cache_path = f'{CACHE_DIR}/{sid}.pt'
    if not os.path.exists(cache_path):
        continue
    d = torch.load(cache_path, weights_only=False)
    metal_counts[d.y.shape[0]] += 1

total_structs  = sum(metal_counts.values())
total_metals   = sum(k * v for k, v in metal_counts.items())
multi_structs  = sum(v for k, v in metal_counts.items() if k > 1)
multi_metals   = sum(k * v for k, v in metal_counts.items() if k > 1)

print(f'{"Metals":>8s}  {"Structures":>12s}')
for k in sorted(metal_counts):
    print(f'{k:>8d}  {metal_counts[k]:>12d}')
print(f'{"─"*22}')
print(f'{"Total":>8s}  {total_structs:>12d}  ({total_metals} metal sites)')
print()
print(f'Single-metal structures: {metal_counts[1]:,}  ({metal_counts[1]/total_structs*100:.1f}%)')
print(f'Multi-metal structures:  {multi_structs:,}  ({multi_structs/total_structs*100:.1f}%)')
print(f'Metal sites in multi:    {multi_metals:,}  (potential additional targets)')


In [ ]:
# [NON-ESSENTIAL] Debug check — inspect zero-metal structures in the raw parquet to diagnose missing metal data
# Check a few zero-metal structure IDs against the raw parquet
zero_metal_ids = [sid for sid in all_ids
                  if os.path.exists(f'{CACHE_DIR}/{sid}.pt')
                  and torch.load(f'{CACHE_DIR}/{sid}.pt', weights_only=False).y.shape[0] == 0]

print(f'Zero-metal cached structures: {len(zero_metal_ids)}')

# Check one in the chunk parquets
import glob, pandas as pd
sid = zero_metal_ids[0]
print(f'\nChecking {sid}:')
for fpath in glob.glob('data/gemmi_atom14/chunk_*.parquet'):
    df = pd.read_parquet(fpath, columns=['structure_id'])
    if sid in df['structure_id'].values:
        df_full = pd.read_parquet(fpath)
        df_sid = df_full[df_full['structure_id'] == sid]
        print(f'  Found in {fpath}: {len(df_sid)} rows, is_metal counts:')
        print(f'  {df_sid["is_metal"].value_counts().to_dict()}')
        print(f'  res_names: {df_sid["res_name"].unique()[:10]}')
        break


In [ ]:
# [NON-ESSENTIAL] One-time: build ESM-2 35M per-residue embedding cache — run once before training with USE_PLM=True
# Requires: pip install fair-esm
# Runtime on MPS/CPU: ~30–60 min for all 43k structures; idempotent if interrupted

import esm as esm_pkg
import torch, os, glob
import pandas as pd
import numpy as np

ESM_CACHE_DIR = 'data/esm2_cache'
REPR_LAYER    = 12   # last layer of esm2_t12_35M_UR50D (12 layers, 480-dim)

os.makedirs(ESM_CACHE_DIR, exist_ok=True)

esm_model, alphabet = esm_pkg.pretrained.esm2_t12_35M_UR50D()
esm_model = esm_model.eval()
batch_converter = alphabet.get_batch_converter()

THREE_TO_ONE = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G',
    'HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S',
    'THR':'T','TRP':'W','TYR':'Y','VAL':'V','UNK':'X',
}

# Build parquet index — reuses CHUNK_GLOB defined in the model cell
esm_index = {}
for fpath in glob.glob(CHUNK_GLOB):
    df_idx = pd.read_parquet(fpath, columns=['structure_id'])
    for sid in df_idx['structure_id'].unique():
        esm_index[sid] = fpath

n_done = n_skip = n_err = 0
for sid in all_ids:
    out_path = os.path.join(ESM_CACHE_DIR, f'{sid}.pt')
    if os.path.exists(out_path):
        n_skip += 1
        continue
    if sid not in esm_index:
        n_err += 1
        continue
    try:
        df = pd.read_parquet(esm_index[sid])
        df = df[(df['structure_id'] == sid) & (~df['is_metal'])].reset_index(drop=True)

        # Apply the same CA-valid filter as _process() so residue count aligns with d.x
        coords14_raw = df[[f'{c}{i}' for i in range(14) for c in ('x', 'y', 'z')]].values.reshape(-1, 14, 3)
        valid    = np.isfinite(coords14_raw[:, 1, :]).all(axis=1)
        res_names = df['res_name'].values[valid]

        seq = ''.join(THREE_TO_ONE.get(r, 'X') for r in res_names)
        if not seq:
            n_err += 1
            continue

        _, _, tokens = batch_converter([(sid, seq)])
        with torch.no_grad():
            results = esm_model(tokens, repr_layers=[REPR_LAYER], return_contacts=False)
        # Strip BOS/EOS tokens → (N_residues, 480)
        emb = results['representations'][REPR_LAYER][0, 1:len(seq) + 1, :].float().cpu()

        torch.save(emb, out_path)
        n_done += 1
    except Exception:
        n_err += 1

    if (n_done + n_skip) % 2000 == 0:
        print(f'done: {n_done}  skipped: {n_skip}  errors: {n_err}')

print(f'Finished. done: {n_done}  skipped: {n_skip}  errors: {n_err}')
print(f'Embeddings written to {ESM_CACHE_DIR}/')


In [ ]:
# [NON-ESSENTIAL] Comparison — model vs. naive CA centroid baseline on the test set
# Respects METAL_FILTER, NUM_HYPOTHESES, and CHECKPOINT set in the training cell.
import torch
from torch_geometric.loader import DataLoader

if METAL_FILTER == 'ZN':
    test_ids_cmp = zn_test_ids
elif METAL_FILTER == 'bare':
    test_ids_cmp = bm_test_ids
elif SINGLE_METAL_ONLY:
    test_ids_cmp = sm_test_ids
else:
    test_ids_cmp = am_test_ids

esm_dir_cmp = ESM_CACHE_DIR if USE_PLM else None
cmp_ds      = MetalBindingDataset(CHUNK_GLOB, test_ids_cmp, cache_dir=CACHE_DIR, esm_cache_dir=esm_dir_cmp)
cmp_loader  = DataLoader(cmp_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model_cmp = MetalPredictionModel(attn_mode=ATTN_MODE, use_plm=USE_PLM, esm_dim=ESM_DIM,
                                  num_hypotheses=NUM_HYPOTHESES).to(device)
model_cmp.load_state_dict(torch.load(CHECKPOINT, weights_only=True))
model_cmp.eval()

naive_errors = []
model_errors = []

use_single = (SINGLE_METAL_ONLY or METAL_FILTER is not None)

with torch.no_grad():
    for batch in cmp_loader:
        batch = batch.to(device)
        num_graphs = batch.batch.max().item() + 1

        # Naive: mean CA position per graph
        counts = torch.zeros(num_graphs, dtype=torch.float, device=device) \
                      .scatter_add(0, batch.batch,
                                   torch.ones(batch.batch.shape[0], dtype=torch.float, device=device))
        centroid = torch.zeros(num_graphs, 3, device=device)
        centroid.scatter_add_(0, batch.batch.unsqueeze(1).expand(-1, 3), batch.pos)
        centroid /= counts.unsqueeze(1)

        esm_feat = batch.esm.to(device) if (USE_PLM and hasattr(batch, 'esm')) else None
        pred_out = model_cmp(batch.x, batch.edge_index, batch.edge_attr,
                             batch.batch, batch.res_type, batch.pos, batch.pos_distal,
                             esm=esm_feat)

        if use_single:
            naive_d = torch.norm(centroid - batch.y, dim=-1)
            if NUM_HYPOTHESES > 1:
                sq = ((pred_out - batch.y.unsqueeze(1)) ** 2).sum(-1)
                model_d = sq.min(dim=1).values.sqrt()
            else:
                model_d = torch.norm(pred_out - batch.y, dim=-1)
            naive_errors.extend(naive_d.cpu().tolist())
            model_errors.extend(model_d.cpu().tolist())
        else:
            y_split = torch.split(batch.y, batch.num_metals.tolist())
            for i, metals in enumerate(y_split):
                naive_errors.append(torch.norm(centroid[i] - metals, dim=-1).min().item())
                if NUM_HYPOTHESES > 1:
                    model_errors.append(
                        torch.norm(pred_out[i] - metals, dim=-1).min().item()
                    )
                else:
                    model_errors.append(torch.norm(pred_out[i] - metals, dim=-1).min().item())

naive_errors = torch.tensor(naive_errors)
model_errors = torch.tensor(model_errors)
wins = (model_errors < naive_errors).float().mean() * 100

print(f'Checkpoint: {CHECKPOINT}')
print(f'Test set:   {len(test_ids_cmp)} structures  |  filter: {METAL_FILTER}  |  hypotheses: {NUM_HYPOTHESES}')
print()
print(f'{"":22s} {"Naive centroid":>16s} {"Model":>10s}')
print(f'{"RMSE":22s} {naive_errors.pow(2).mean().sqrt():>14.2f} Å {model_errors.pow(2).mean().sqrt():>8.2f} Å')
print(f'{"Mean error":22s} {naive_errors.mean():>14.2f} Å {model_errors.mean():>8.2f} Å')
print(f'{"Median error":22s} {naive_errors.median():>14.2f} Å {model_errors.median():>8.2f} Å')
print(f'{"Error < 5 Å":22s} {(naive_errors < 5).float().mean()*100:>13.1f}% {(model_errors < 5).float().mean()*100:>8.1f}%')
print(f'{"Error < 10 Å":22s} {(naive_errors < 10).float().mean()*100:>13.1f}% {(model_errors < 10).float().mean()*100:>8.1f}%')
print()
print(f'Model beats naive on {wins:.1f}% of structures')

In [ ]:
# [NON-ESSENTIAL] Error distribution analysis — percentile breakdown and worst-case outlier inspection (naive vs. model)
import numpy as np

# Error distribution breakdown: naive centroid vs model
percentiles = [10, 25, 50, 75, 90, 95, 99]
print(f'{"Percentile":>12s}  {"Naive centroid":>16s}  {"Model":>10s}  {"Model wins?":>12s}')
print('─' * 56)
for p in percentiles:
    n = np.percentile(naive_errors.numpy(), p)
    m = np.percentile(model_errors.numpy(), p)
    flag = '✓' if m < n else '✗'
    print(f'{p:>11d}%  {n:>14.2f} Å  {m:>8.2f} Å  {flag:>12s}')

# Worst naive-centroid outliers
print(f'\nTop 15 naive centroid outliers:')
print(f'{"Rank":>5s}  {"Naive err":>10s}  {"Model err":>10s}  {"Delta":>8s}')
print('─' * 40)
sorted_idx = naive_errors.argsort(descending=True)
for rank, i in enumerate(sorted_idx[:15].tolist()):
    n, m = naive_errors[i].item(), model_errors[i].item()
    delta = m - n
    flag = f'({delta:+.1f})'
    print(f'{rank+1:>5d}  {n:>8.2f} Å  {m:>8.2f} Å  {flag:>8s}')

# Structures where model is much worse than centroid
delta = model_errors - naive_errors
print(f'\nStructures where model is >5 Å worse than centroid: {(delta > 5).sum().item()}')
print(f'Structures where model is >5 Å better than centroid: {(delta < -5).sum().item()}')
print(f'Structures where model is within 1 Å of centroid: {(delta.abs() < 1).sum().item()}')


In [ ]:
# [NON-ESSENTIAL] Install matplotlib (run once in a fresh environment)
! pip install matplotlib

In [ ]:
# [NON-ESSENTIAL] Visualization — mean attention weight by residue type, highlighting known metal-binding residues (HIS, CYS, ASP, GLU)
import matplotlib.pyplot as plt
from collections import defaultdict

model.load_state_dict(torch.load(CHECKPOINT, weights_only=True))
model.eval()

attn_by_res = defaultdict(list)
IDX_TO_RES  = {i: r for r, i in RES_TO_IDX.items()}

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        esm_feat = batch.esm.to(device) if (USE_PLM and hasattr(batch, 'esm')) else None
        _, attn = model(batch.x, batch.edge_index, batch.edge_attr,
                        batch.batch, batch.res_type, batch.pos, batch.pos_distal,
                        return_attn=True, esm=esm_feat)
        for res_idx, weight in zip(batch.res_type.cpu().tolist(), attn.cpu().tolist()):
            attn_by_res[IDX_TO_RES[res_idx]].append(weight)

mean_attn = {r: np.mean(ws) for r, ws in attn_by_res.items()}
sorted_res = sorted(mean_attn, key=mean_attn.get, reverse=True)

METAL_BINDERS = {'HIS', 'CYS', 'ASP', 'GLU'}
colors = ['tomato' if r in METAL_BINDERS else 'steelblue' for r in sorted_res]

plt.figure(figsize=(12, 4))
plt.bar(sorted_res, [mean_attn[r] for r in sorted_res], color=colors)
plt.xlabel('Residue type')
plt.ylabel('Mean attention weight')
plt.title('Mean attention weight by residue type\n(red = known metal binders)')
plt.tight_layout()
plt.show()

print('Top 5 residues by attention:')
for r in sorted_res[:5]:
    print(f'  {r}: {mean_attn[r]:.6f}')


In [ ]:
# [NON-ESSENTIAL] Exploratory — count and rank metal residue types (e.g. ZN, MG, CA) across all parquet chunks
import glob
import pandas as pd
from collections import Counter

metal_residues = Counter()

for fpath in glob.glob('data/gemmi_atom14/chunk_*.parquet'):
    df = pd.read_parquet(fpath, columns=['res_name', 'is_metal'])
    metal_residues.update(df[df['is_metal']]['res_name'].tolist())

print(f'Unique metal residue names in parquet: {len(metal_residues)}\n')
for name, count in metal_residues.most_common():
    print(f'  {name:6s}  {count:8d}')


In [ ]:
# [NON-ESSENTIAL] Empty cell
